In [1]:
import torch 
print(f"GPU: {torch.cuda.get_device_name(0)}") 

GPU: Tesla T4


In [3]:
import pandas as pd 
import numpy as np 
import torch 
import csv 
import os 
from transformers import AutoTokenizer, AutoModel 
from torch.optim import AdamW 
from torch.utils.data import Dataset, DataLoader 
from transformers import get_linear_schedule_with_warmup 
from scipy.stats import spearmanr 
from sklearn.metrics import mean_squared_error 
DATA_ROOT   = '/kaggle/input/datasets/resegomorei/semrel2024-raw/raw' 
RESULTS_LOG = '/kaggle/working/results_log.csv' 
CKPT_DIR    = '/kaggle/working/checkpoints' 
MODEL_NAME  = 'castorini/afriberta_large' 
MAX_LEN     = 128 
BATCH_SIZE  = 32  # AfriBERTa is smaller, can use larger batch 
EPOCHS      = 10 
LR          = 2e-5 
PATIENCE    = 3 
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
 
os.makedirs(CKPT_DIR, exist_ok=True) 
print(f"Device: {DEVICE}") 
print(f"Model:  {MODEL_NAME}") 

Device: cuda
Model:  castorini/afriberta_large


In [4]:
class SemRelDataset(Dataset): 
    def __init__(self, df, tokenizer, max_len): 
        self.df        = df.reset_index(drop=True) 
        self.tokenizer = tokenizer 
        self.max_len   = max_len 
 
    def __len__(self): 
        return len(self.df) 
 
    def __getitem__(self, idx): 
        row = self.df.iloc[idx] 
        enc = self.tokenizer( 
            str(row['sentence1']), 
            str(row['sentence2']), 
            truncation=True, 
            max_length=self.max_len, 
            padding='max_length', 
            return_tensors='pt' 
        ) 
        return { 
            'input_ids':      enc['input_ids'].squeeze(), 
            'attention_mask': enc['attention_mask'].squeeze(), 
            'label':          torch.tensor(row['label'], dtype=torch.float) 
        } 
 
print("SemRelDataset defined.") 

SemRelDataset defined.


In [5]:
class AfriBERTaModel(torch.nn.Module): 
    def __init__(self, model_name): 
        super().__init__() 
        self.encoder   = AutoModel.from_pretrained(model_name) 
        self.regressor = torch.nn.Linear(self.encoder.config.hidden_size, 1) 
        self.dropout   = torch.nn.Dropout(0.1) 
 
    def forward(self, input_ids, attention_mask): 
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask) 
        cls = outputs.last_hidden_state[:, 0, :] 
        return self.regressor(self.dropout(cls)).squeeze(-1) 
 
 
def evaluate(model, loader): 
    model.eval() 
    preds, labels = [], [] 
    with torch.no_grad(): 
        for batch in loader: 
            ids  = batch['input_ids'].to(DEVICE) 
            mask = batch['attention_mask'].to(DEVICE) 
            out  = model(ids, mask).cpu().numpy() 
            preds.extend(out) 
            labels.extend(batch['label'].numpy()) 
    rho, _ = spearmanr(preds, labels) 
    mse    = mean_squared_error(labels, preds) 
    return rho, mse, preds 
 
 
def train(lang_code, lang_name, tokenizer): 
    print(f"\n{'='*60}") 
    print(f"  Training AfriBERTa on {lang_name}") 
    print(f"{'='*60}") 
 
    train_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv') 
    dev_df   = pd.read_csv(f'{DATA_ROOT}/{lang_code}/dev.csv') 
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv') 
 
    train_loader = DataLoader(SemRelDataset(train_df, tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE, shuffle=True) 
    dev_loader   = DataLoader(SemRelDataset(dev_df,   tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE) 
    test_loader  = DataLoader(SemRelDataset(test_df,  tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE) 
 
    model     = AfriBERTaModel(MODEL_NAME).to(DEVICE) 
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01) 
    total_steps = len(train_loader) * EPOCHS 
    scheduler = get_linear_schedule_with_warmup( 
        optimizer, 
        num_warmup_steps=int(0.1 * total_steps), 
        num_training_steps=total_steps 
    ) 
    loss_fn = torch.nn.MSELoss() 
 
    best_rho, patience_counter, best_epoch = -1, 0, 0 
 
    for epoch in range(1, EPOCHS + 1): 
        model.train() 
        total_loss = 0 
        for batch in train_loader: 
            ids   = batch['input_ids'].to(DEVICE) 
            mask  = batch['attention_mask'].to(DEVICE) 
            lbls  = batch['label'].to(DEVICE) 
            optimizer.zero_grad() 
            preds = model(ids, mask) 
            loss  = loss_fn(preds, lbls) 
            loss.backward() 
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
            optimizer.step() 
            scheduler.step() 
            total_loss += loss.item() 
 
        dev_rho, dev_mse, _ = evaluate(model, dev_loader) 
        avg_loss = total_loss / len(train_loader) 
        print(f"  Epoch {epoch:2d} | loss={avg_loss:.4f} | dev ρ={dev_rho:.4f} | dev MSE={dev_mse:.4f}")
 
        if dev_rho > best_rho: 
            best_rho = dev_rho 
            best_epoch = epoch 
            patience_counter = 0 
            torch.save(model.state_dict(), 
                       f'{CKPT_DIR}/afriberta_{lang_code}_best.pt') 
            print(f"           ✓ New best saved (ρ={best_rho:.4f})") 
        else: 
            patience_counter += 1 
            if patience_counter >= PATIENCE: 
                print(f"  Early stopping at epoch {epoch} (patience={PATIENCE})") 
                break 
 
    model.load_state_dict(torch.load(f'{CKPT_DIR}/afriberta_{lang_code}_best.pt')) 
    test_rho, test_mse, _ = evaluate(model, test_loader) 
    print(f"\n  Best epoch: {best_epoch} | Test ρ={test_rho:.4f} | Test MSE={test_mse:.4f}") 
    return test_rho, test_mse 
 
print("AfriBERTa functions defined.")

AfriBERTa functions defined.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) 
 
for lang_code, lang_name in [('eng','English'), ('hau','Hausa'), ('kin','Kinyarwanda')]: 
    rho, mse = train(lang_code, lang_name, tokenizer) 
 
    with open(RESULTS_LOG, 'a', newline='') as f: 
        writer = csv.DictWriter(f, fieldnames=[ 
            'experiment_id','model','model_variant','language', 
            'split','spearman_rho','mse','augmented','notes' 
        ]) 
        writer.writerow({ 
            'experiment_id': f'TL-4-{lang_code}', 
            'model': 'afriberta_finetuned', 
            'model_variant': 'castorini/afriberta_large', 
            'language': lang_name, 
            'split': 'test', 
            'spearman_rho': round(rho, 4), 
            'mse': round(mse, 4), 
            'augmented': False, 
            'notes': f'AfriBERTa fine-tuned on {lang_name} train set.' 
        }) 
    print(f"✓ {lang_name} logged: ρ={rho:.4f}, MSE={mse:.4f}") 

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


  Training AfriBERTa on English


...

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch  1 | loss=0.2880 | dev ρ=0.3349 | dev MSE=0.0687
           ✓ New best saved (ρ=0.3349)
  Epoch  2 | loss=0.0965 | dev ρ=0.4564 | dev MSE=0.0667
           ✓ New best saved (ρ=0.4564)
  Epoch  3 | loss=0.0795 | dev ρ=0.5312 | dev MSE=0.0612
           ✓ New best saved (ρ=0.5312)
  Epoch  4 | loss=0.0651 | dev ρ=0.5290 | dev MSE=0.0547
  Epoch  5 | loss=0.0595 | dev ρ=0.5903 | dev MSE=0.0707
           ✓ New best saved (ρ=0.5903)
  Epoch  6 | loss=0.0526 | dev ρ=0.5698 | dev MSE=0.0487
  Epoch  7 | loss=0.0463 | dev ρ=0.5689 | dev MSE=0.0516
  Epoch  8 | loss=0.0432 | dev ρ=0.5772 | dev MSE=0.0507
  Early stopping at epoch 8 (patience=3)

  Best epoch: 5 | Test ρ=0.4756 | Test MSE=0.0807
✓ Hausa logged: ρ=0.4756, MSE=0.0807

  Training AfriBERTa on Kinyarwanda


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch  1 | loss=0.1511 | dev ρ=0.1399 | dev MSE=0.1063
           ✓ New best saved (ρ=0.1399)
  Epoch  2 | loss=0.0747 | dev ρ=0.1841 | dev MSE=0.0656
           ✓ New best saved (ρ=0.1841)
  Epoch  3 | loss=0.0561 | dev ρ=0.4460 | dev MSE=0.0579
           ✓ New best saved (ρ=0.4460)
  Epoch  4 | loss=0.0539 | dev ρ=0.4922 | dev MSE=0.0527
           ✓ New best saved (ρ=0.4922)
  Epoch  5 | loss=0.0454 | dev ρ=0.5328 | dev MSE=0.0529
           ✓ New best saved (ρ=0.5328)
  Epoch  6 | loss=0.0390 | dev ρ=0.5481 | dev MSE=0.0496
           ✓ New best saved (ρ=0.5481)
  Epoch  7 | loss=0.0351 | dev ρ=0.5539 | dev MSE=0.0504
           ✓ New best saved (ρ=0.5539)
  Epoch  8 | loss=0.0324 | dev ρ=0.5267 | dev MSE=0.0584
  Epoch  9 | loss=0.0312 | dev ρ=0.5416 | dev MSE=0.0476
  Epoch 10 | loss=0.0282 | dev ρ=0.5410 | dev MSE=0.0474
  Early stopping at epoch 10 (patience=3)

  Best epoch: 7 | Test ρ=0.4926 | Test MSE=0.0366
✓ Kinyarwanda logged: ρ=0.4926, MSE=0.0366


In [7]:
def evaluate_zero_shot_afriberta(lang_code, lang_name): 
    print(f"\nAfriBERTa zero-shot on {lang_name}...") 
    test_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv') 
    test_loader = DataLoader( 
        SemRelDataset(test_df, tokenizer, MAX_LEN), 
        batch_size=BATCH_SIZE 
    ) 
    model = AfriBERTaModel(MODEL_NAME).to(DEVICE) 
    model.load_state_dict(torch.load(f'{CKPT_DIR}/afriberta_eng_best.pt')) 
    rho, mse, _ = evaluate(model, test_loader) 
    print(f"  {lang_name}: ρ={rho:.4f}, MSE={mse:.4f}") 
 
    with open(RESULTS_LOG, 'a', newline='') as f: 
        writer = csv.DictWriter(f, fieldnames=[ 
            'experiment_id','model','model_variant','language', 
            'split','spearman_rho','mse','augmented','notes' 
        ]) 
        writer.writerow({ 
            'experiment_id': f'TL-3-{lang_code}', 
            'model': 'afriberta_zeroshot', 
            'model_variant': 'castorini/afriberta_large', 
            'language': lang_name, 
            'split': 'test', 
            'spearman_rho': round(rho, 4), 
            'mse': round(mse, 4), 
            'augmented': False, 
            'notes': 'AfriBERTa zero-shot: English fine-tuned checkpoint on African language.' 
        }) 
    return rho, mse 
 
# Zero-shot on all three African languages 
for lang_code, lang_name in [('afr','Afrikaans'), ('hau','Hausa'), ('kin','Kinyarwanda')]: 
    evaluate_zero_shot_afriberta(lang_code, lang_name) 


AfriBERTa zero-shot on Afrikaans...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Afrikaans: ρ=0.5354, MSE=0.1181

AfriBERTa zero-shot on Hausa...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Hausa: ρ=0.3108, MSE=0.0724

AfriBERTa zero-shot on Kinyarwanda...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Kinyarwanda: ρ=0.3524, MSE=0.0496


In [9]:
import pandas as pd

LOG_HEADERS = ['experiment_id','model','model_variant','language',
               'split','spearman_rho','mse','augmented','notes']

# Read the headerless CSV and assign the proper column names
df = pd.read_csv('/kaggle/working/results_log.csv', header=None, names=LOG_HEADERS)

# Rewrite the file with headers so the downloaded copy is clean
df.to_csv('/kaggle/working/results_log.csv', index=False)

print(f"\n✓ {len(df)} results saved to results_log.csv")
print(df[['experiment_id','language','spearman_rho','mse']].to_string(index=False))

# This file will be available in Output panel for download
print("\n✓ Download results_log.csv from the Output panel on the right.")


✓ 6 results saved to results_log.csv
experiment_id    language  spearman_rho    mse
     TL-4-eng     English        0.6561 0.0292
     TL-4-hau       Hausa        0.4756 0.0807
     TL-4-kin Kinyarwanda        0.4926 0.0366
     TL-3-afr   Afrikaans        0.5354 0.1181
     TL-3-hau       Hausa        0.3108 0.0724
     TL-3-kin Kinyarwanda        0.3524 0.0496

✓ Download results_log.csv from the Output panel on the right.
